# 🔮 Comprehensive Predictions — Churn, Spending & Segmentation

This notebook runs all three models against the test set and produces a unified results file.

| Model | Task | Output |
|---|---|---|
| `best_model_churn.pkl` | Classification | Churn prediction + probability |
| `regression_model.pkl` | Regression | Estimated spending (DT) |
| `kmeans_model.pkl` | Clustering | Customer segment |

>  **Prerequisites** — run these notebooks first:
> 1. `preprocessing_pipeline.ipynb`
> 2. `train_classification.ipynb`
> 3. Regression training notebook

##  Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings

warnings.filterwarnings('ignore', category=UserWarning)
print('✅ Imports OK')

✅ Imports OK


##  Configuration

In [2]:
MODEL_PATHS = {
    'classifier': '../models/best_model_churn.pkl',
    'regressor':  '../models/regression_model.pkl',
    'kmeans':     '../models/kmeans_model.pkl',
}

DATA_TEST_PATH   = '../data/train_test/X_test.csv'
DATA_TEST_REG_PATH = '../data/regression_specific/X_test_reg.csv'  # For regression model
TARGET_TEST_PATH = '../data/train_test/y_test.csv'
OUTPUT_PATH      = '../data/results/test_predictions_complet.csv'

# Customer segment labels — adjust based on your KMeans analysis
SEGMENT_MAP = {
    0: 'Stable',
    1: 'VIP',
    2: 'Occasionnel',
    3: 'À Risque',
}

##  1. Load Models

In [3]:
models = {}

for key, path in MODEL_PATHS.items():
    if os.path.exists(path):
        models[key] = joblib.load(path)
        print(f"✅ '{key}' loaded — {type(models[key]).__name__}")
    else:
        print(f"⚠️  '{key}' not found at {path} — this block will be skipped.")

print(f"\n{len(models)}/{len(MODEL_PATHS)} models available.")

✅ 'classifier' loaded — XGBClassifier
✅ 'regressor' loaded — RandomForestRegressor
✅ 'kmeans' loaded — KMeans

3/3 models available.


##  2. Load Test Data

In [4]:
assert os.path.exists(DATA_TEST_PATH), (
    f"❌ {DATA_TEST_PATH} not found — run preprocessing_pipeline.ipynb first."
)

# Load BOTH test sets (different feature types, different sample sizes)
X_test = pd.read_csv(DATA_TEST_PATH)  # PCA features - 822 samples (for classifier & kmeans)
X_test_reg = pd.read_csv(DATA_TEST_REG_PATH)  # Original features - 602 samples (for regressor)

print(f"✅ Test set (PCA): {len(X_test)} customers × {X_test.shape[1]} components")
print(f"✅ Test set (Regression): {len(X_test_reg)} customers × {X_test_reg.shape[1]} features")

# Load ground truth if available
y_true = None
if os.path.exists(TARGET_TEST_PATH):
    y_true = pd.read_csv(TARGET_TEST_PATH).values.ravel()
    print(f"✅ Ground truth loaded — Churn rate: {y_true.mean():.2%}")
else:
    print("⚠️  Ground truth (y_test) not found — Real_Status column will be omitted.")

print(f"\nℹ️  Classifier & Clustering will use {len(X_test)} samples")
print(f"ℹ️  Regression will predict on first {len(X_test_reg)} samples only")
X_test.head()

✅ Test set (PCA): 822 customers × 10 components
✅ Test set (Regression): 602 customers × 31 features
✅ Ground truth loaded — Churn rate: 34.43%

ℹ️  Classifier & Clustering will use 822 samples
ℹ️  Regression will predict on first 602 samples only


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,-1.859450,-0.108343,0.141432,0.554251,1.150383,-0.890124,0.618946,0.460643,0.812959,0.635631
1,1.573018,-0.424638,-1.162225,-0.112055,1.054218,0.619287,-1.084592,-1.183602,-1.232167,0.883807
2,0.505772,-0.718953,-0.350930,-0.765851,-0.393811,-4.623587,0.443059,-1.473900,-0.322109,-0.113752
3,-1.760199,-2.157234,0.576520,-0.889547,0.809588,-0.567612,1.082044,1.317984,-0.232085,1.165040
4,1.573551,-1.692101,1.612323,-1.116326,-0.465957,0.986412,-2.765707,-0.364121,-0.909312,-1.038849


##  3. Classification — Churn Prediction

In [5]:
results = pd.DataFrame(index=range(len(X_test)))

if 'classifier' in models:
    results['Predicted_Churn']       = models['classifier'].predict(X_test)
    results['Churn_Probability_%']   = (
        models['classifier'].predict_proba(X_test)[:, 1] * 100
    ).round(2)

    churn_count = results['Predicted_Churn'].sum()
    print(f"✅ Classification done.")
    print(f"   Predicted churn : {churn_count} / {len(results)} ({churn_count/len(results):.2%})")
    results[['Predicted_Churn', 'Churn_Probability_%']].describe()
else:
    print("⏭️  Classifier not available — skipping.")

✅ Classification done.
   Predicted churn : 285 / 822 (34.67%)


##  4. Regression — Estimated Spending

In [6]:
if 'regressor' in models:
    # Predict on regression test data (602 samples)
    raw_preds = models['regressor'].predict(X_test_reg)
    
    # Create full-length array to match all 822 customers (fill extra with NaN)
    spending_preds = np.full(len(X_test), np.nan)
    spending_preds[:len(X_test_reg)] = raw_preds

    # Auto-detect log scale vs real scale based on mean value (ignoring NaNs)
    if np.nanmean(spending_preds) < 20:
        print("   ℹ️  Log-scale predictions detected — applying np.expm1()")
        results['Predicted_Spending_DT'] = np.expm1(spending_preds).round(2)
    else:
        results['Predicted_Spending_DT'] = spending_preds.round(2)

    print(f"✅ Regression done.")
    valid_preds = spending_preds[~np.isnan(spending_preds)]
    print(f"   Predictions: {len(valid_preds)} customers | {len(X_test) - len(valid_preds)} N/A")
    print(f"   Mean predicted spending : {np.nanmean(results['Predicted_Spending_DT']):.2f} DT")
    print(f"   Total financial potential: {np.nansum(results['Predicted_Spending_DT']):.2f} DT")
    results['Predicted_Spending_DT'].describe()
else:
    print("⏭️  Regressor not available — skipping.")

   ℹ️  Log-scale predictions detected — applying np.expm1()
✅ Regression done.
   Predictions: 602 customers | 220 N/A
   Mean predicted spending : 1469.23 DT
   Total financial potential: 884477.34 DT


##  5. Clustering — Customer Segmentation

In [7]:
if 'kmeans' in models:
    # Use .values to avoid feature name warnings
    results['Customer_Segment'] = models['kmeans'].predict(X_test.values)
    results['Segment_Name']     = results['Customer_Segment'].map(SEGMENT_MAP)

    print(f"✅ Clustering done.")
    print("\nSegment distribution:")
    print(results['Segment_Name'].value_counts())
else:
    print("⏭️  KMeans model not available — skipping.")

✅ Clustering done.

Segment distribution:
Segment_Name
VIP            375
À Risque       204
Stable         150
Occasionnel     93
Name: count, dtype: int64


##  6. Add Ground Truth & Preview Results

In [8]:
if y_true is not None:
    results['Real_Status'] = y_true
    print("✅ Ground truth column added.")

print(f"\nResults shape: {results.shape}")
results.head(10)

✅ Ground truth column added.

Results shape: (822, 6)


,Predicted_Churn,Churn_Probability_%,Predicted_Spending_DT,Customer_Segment,Segment_Name,Real_Status
0,1,99.930000,101.02,1,VIP,1
1,0,0.070000,1901.95,0,Stable,0
2,0,17.520000,324.97,1,VIP,0
3,1,99.629997,650.97,1,VIP,1
4,0,1.600000,1704.02,0,Stable,0
5,1,99.870003,103.02,3,À Risque,1
6,1,76.870003,5551.43,1,VIP,0
7,0,0.790000,2635.72,3,À Risque,0
8,1,59.590000,324.38,3,À Risque,1
9,0,18.440001,1422.03,3,À Risque,0


##  7. Export Results

In [9]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
results.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Results saved to: {OUTPUT_PATH}")

✅ Results saved to: ../data/results/test_predictions_complet.csv


##  8. Summary

In [10]:
print("=" * 60)
print("         PREDICTION SUMMARY")
print("=" * 60)
print(f"  Customers analysed : {len(results)}")

if 'Predicted_Churn' in results.columns:
    n = results['Predicted_Churn'].sum()
    print(f"  Churn detected     : {n} ({n/len(results):.2%})")

if 'Predicted_Spending_DT' in results.columns:
    print(f"  Avg. spending      : {results['Predicted_Spending_DT'].mean():.2f} DT")
    print(f"  Total potential    : {results['Predicted_Spending_DT'].sum():.2f} DT")

if 'Segment_Name' in results.columns:
    print("\n  Segment breakdown:")
    for seg, count in results['Segment_Name'].value_counts().items():
        print(f"    {seg:<15} : {count}")

print("="*60)
print(f"  Output → {OUTPUT_PATH}")
print("="*60)

         PREDICTION SUMMARY
  Customers analysed : 822
  Churn detected     : 285 (34.67%)
  Avg. spending      : 1469.23 DT
  Total potential    : 884477.34 DT

  Segment breakdown:
    VIP             : 375
    À Risque        : 204
    Stable          : 150
    Occasionnel     : 93
  Output → ../data/results/test_predictions_complet.csv
